## Stochastic Optimization

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q manipulation[grader]

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import mpld3
import numpy as np
from pydrake.all import (
    BaseField,
    Evaluate,
    Fields,
    PointCloud,
    Rgba,
    RigidTransform,
    Sphere,
    Variable,
)

from manipulation import running_as_notebook
from manipulation.meshcat_utils import StartMeshcat

if running_as_notebook:
    mpld3.enable_notebook()

In [ ]:
def loss(theta):
    x = theta[0]
    y = theta[1]
    eval = 2 * x**2 - 1.05 * x**4 + x**6 / 6 + x * y + y**2
    return 0.25 * eval


def generate_color_mat(color_vec, shape):
    color_mat = np.tile(
        np.array(color_vec).astype(np.float32).reshape(3, 1), (1, shape[1])
    )
    return color_mat


def visualize_loss(
    meshcat,
    loss,
    colormap="viridis",
    spacing=0.01,
    clip_min=None,
    clip_max=None,
):
    # Create a grid of thetas and evaluate losses.
    points = []
    for i in np.arange(-3, 3, spacing):
        for j in np.arange(-3, 3, spacing):
            points.append([i, j, loss(np.array([i, j]))])
    points = np.array(points)

    # Normalize losses and color them according to colormap.
    cmap = matplotlib.colormaps[colormap]
    min_loss = np.min(points[:, 2]) if clip_min == None else clip_min
    max_loss = np.max(points[:, 2]) if clip_max == None else clip_max

    colors = []
    for i in range(points.shape[0]):
        normalized_loss = (points[i, 2] - min_loss) / (max_loss - min_loss)
        colors.append(list(cmap(normalized_loss))[0:3])

    cloud = PointCloud(points.shape[0], Fields(BaseField.kXYZs | BaseField.kRGBs))
    cloud.mutable_xyzs()[:] = points.T
    cloud.mutable_rgbs()[:] = 255 * np.array(colors).T

    meshcat.Delete()
    meshcat.SetProperty("/Background", "visible", False)
    meshcat.SetObject("/loss", cloud, point_size=0.03)


def visualize_trajectory(trajectory):
    points = PointCloud(trajectory.shape[0])
    points.mutable_xyzs()[:] = trajectory.T
    meshcat.SetObject("/traj", points, rgba=Rgba(1, 0, 0), point_size=0.03)
    meshcat.SetLine("/traj_line", trajectory.T, rgba=Rgba(1, 0, 0))

    # Visualize the initial guess.
    meshcat.SetObject("/traj_initial", Sphere(0.05), Rgba(1, 0, 0))
    meshcat.SetTransform("/traj_initial", RigidTransform(trajectory[0, :]))

    # Visualize the final point of the iteration.
    meshcat.SetObject("/traj_final", Sphere(0.05), Rgba(0, 1, 0))
    meshcat.SetTransform("/traj_final", RigidTransform(trajectory[-1, :]))

In [ ]:
# Start the visualizer.
meshcat = StartMeshcat()

## The Three Hump Camel 
In this exercise, we'll implement our own versions of gradient descent and stochastic gradient descent! 

Our goal is to find the minima of the following function:

$$l(x)=\frac{1}{4}\bigg(2x_1^2-1.05x_1^4+\frac{x_1^6}{6}+x_1x_2+x_2^2\bigg)$$

Note: this function is defined above as `loss(x)`.

We have visualized the landscape of this function in meshcat if you run the cell below! You will notice the following things:

1. This function has 3 local minima (hence, the name 'three hump camel')
2. The global minima is located at $f([0,0])=0$. 

In [ ]:
# The parameters are optimized for best visualization in meshcat.
# For faster visualization, try increasing spacing.
visualize_loss(meshcat, loss, colormap="viridis", spacing=0.02, clip_max=2.0)

## Gradient Descent

As we saw in the lecture, one way of trying to find the minimum of $l(x)$ is to use explicit gradients and do gradient descent. 

$$x \leftarrow x - \eta\bigg(\frac{\partial l(x)}{\partial x}\bigg)^T$$

We've set up a basic outline of the gradient descent algorithm for you. Take a look at the following function `gradient_descent` that implements the following steps:

1. Initialize $x\in\mathbb{R}^2$ at random from some bounded region.
2. Until maximum iteration, update $x$ according to some update rule like the one defined above. 

Throughout the following notebook, we will walk-through a handful of potential update functions.

In [ ]:
def gradient_descent(rate, update_rule, initial_x=None, iter=1000):
    """gradient descent algorithm
    @params:
    - rate (float): eta variable of gradient descent.
    - update_rule: a function with a signature update_rule(x, rate).
    - initial_x: initial position for gradient descent.
    - iter: number of iterations to run gradient descent for.
    """
    # If no initial guess is supplied, then randomly choose one.
    if initial_x is None:
        x = -3 + 6.0 * np.random.rand(2)
    else:
        x = initial_x
    # Compute loss for first parameter for visualization.
    x_list = []
    x_list.append([x[0], x[1], loss(x)])
    # Loop through with gradient descent.
    for i in range(iter):
        # Update the parameters using update rule.
        x = update_rule(x, rate)
        x_list.append([x[0], x[1], loss(x)])
    return np.array(x_list)

## Deterministic Exact Gradients

**Problem 11.1.a** [2 pts]: Let's first use the standard gradient descent algorithm with exact gradients. Below, you must implement the following simple update function:

$$x \leftarrow x - \eta\bigg(\frac{\partial l(x)}{\partial x}\bigg)^T$$

HINT: You can write down the gradient yourself, but remember you can also use Drake's symbolic differentiation!


In [ ]:
def exact_gradient(x, rate):
    """
    Update rule. Receive theta and update it with the next theta.
    Input:
        - x: input variable x.
        - rate: rate of descent, variable "eta".
    Output:
        - x: updated variable x.
    """

    # YOUR CODE HERE

    return x

When you've completed the function, you can run the below cell to check the visualization! For this problem, the visualization has the following convention:
- Red sphere is the initial guess 
- Green sphere is the final point after `iter` iterations. 
- Every updated parameter is drawn as smaller red cubes. 

In [ ]:
# Compute the trajectory.
trajectory = gradient_descent(0.1, exact_gradient)
visualize_trajectory(trajectory)

If you've implemented it correctly, run the cell multiple times to see the behavior of gradient descent from different initial conditions. 

**Problem 11.1.b** [1 pts] What do you notice about the behaviour of gradient descent given different starting points? When does it converge or not converge (if ever) to the global minimum? 

## Gaussian Gradient Estimators

**Problem 11.1.c** [2 pts]: We can estimate a gradient using only function evaluations (a *zeroth-order* estimator). For $w\sim\mathcal{N}(0,\sigma^2 I)$, define the Gaussian-smoothed loss

$$l_\sigma(x)=\mathbb{E}_w[l(x+w)].$$

The estimator

$$g(x,w)=\frac{l(x+w)-l(x)}{\sigma^2}w$$

is unbiased for $\nabla l_\sigma(x)$, which generally differs from $\nabla l(x)$ at finite $\sigma$. For a smooth loss, its expectation approaches the original gradient as $\sigma\to0$. Smoothing changes the objective; sampling introduces variance around its gradient.

Implement the update

$$x\leftarrow x-\eta g(x,w),\qquad w\sim\mathcal{N}(0,0.25 I).$$

Use `np.random.normal(scale=0.5, size=2)`: NumPy takes the standard deviation, while the denominator is the variance $\sigma^2=0.25$, not the sampled squared norm $\|w\|^2$.

In [ ]:
def approximated_gradient(x, rate):
    """
    Update rule. Receive theta and update it with the next theta.
    Input:
        - x: input variable x.
        - rate: rate of descent, variable "eta".
    Output:
        - x: updated variable x.
    """

    # YOUR CODE HERE

    return x

Again, once you've implemented the function, run the below cell to visualize the trajectory.

In [ ]:
np.random.seed(7)
trajectory = gradient_descent(
    0.0025, approximated_gradient, initial_x=np.array([2.0, -1.0]), iter=10000
)
visualize_trajectory(trajectory)

Compare this trajectory with exact gradient descent from the same initial point. A stochastic trajectory can leave a local basin, but that is not a special advantage of avoiding derivatives: first-order methods with injected noise can also do this. Moreover, Gaussian smoothing can change the local minima themselves. Neither estimator guarantees convergence to the global minimum, and a fixed learning rate can leave persistent fluctuations.

The comparison below separates these effects. The comparison reuses your exact gradient update from part (a) and provides the Gaussian smoothing correction for this polynomial loss:

- **Exact:** $\nabla l(x)$, for the original objective.
- **Smoothed exact:** $\nabla l_\sigma(x)$, with smoothing but no sampling noise.
- **First-order sampled:** $\nabla l(x+w)$, a pathwise estimator of $\nabla l_\sigma(x)$.
- **Zeroth-order:** your estimator from part (c), with the same mean as the pathwise estimator.
- **Antithetic:** $[l(x+w)-l(x-w)]w/(2\sigma^2)$, another estimator of the same smoothed gradient using paired perturbations.
- **Exact + noise:** $\nabla l(x)+\xi$, where $\xi\sim\mathcal{N}(0,I)$, with mean $\nabla l(x)$; this adds noise without smoothing the objective.

Run several seeds and vary the learning rate and iteration count. Which methods leave the initial basin? Compare the two deterministic methods to isolate smoothing, then compare the stochastic estimators with their corresponding deterministic mean. The noise scales and computational costs differ, so these trajectories are an illustration, not a ranking of algorithms. Constant-step noisy methods need not settle at a minimum.

In [ ]:
def loss_gradient(x):
    # Recover the gradient from your update rule in part (a).
    return x - exact_gradient(x, 1.0)


def smoothed_loss_gradient(x, sigma=0.5):
    # Gaussian moments give this correction to the original gradient.
    x1 = x[0]
    correction = 0.25 * (10 * x1**3 * sigma**2 + (15 * sigma**4 - 12.6 * sigma**2) * x1)
    return loss_gradient(x) + np.array([correction, 0.0])


def pathwise_update(x, rate):
    w = np.random.normal(scale=0.5, size=2)
    return x - rate * loss_gradient(x + w)


def antithetic_update(x, rate):
    sigma = 0.5
    w = np.random.normal(scale=sigma, size=2)
    return x - rate * (loss(x + w) - loss(x - w)) * w / (2 * sigma**2)


def noisy_exact_update(x, rate):
    return x - rate * (loss_gradient(x) + np.random.normal(size=2))


initial_x = np.array([2.0, -1.0])
updates = {
    "Exact": exact_gradient,
    "Smoothed exact": lambda x, rate: x - rate * smoothed_loss_gradient(x),
    "First-order sampled": pathwise_update,
    "Zeroth-order": approximated_gradient,
    "Antithetic": antithetic_update,
    "Exact + noise": noisy_exact_update,
}
fig, ax = plt.subplots()
for label, update in updates.items():
    np.random.seed(7)
    trajectory = gradient_descent(0.0025, update, initial_x=initial_x, iter=10000)
    ax.plot(trajectory[:, 2], label=label)
ax.set(xlabel="Iteration", ylabel="Original loss l(x)")
ax.legend()
plt.show()

## Baselines

**Problem 11.1.d** [4 pts]: Consider a scalar baseline $b(x)$ that does not depend on the current perturbation $w$:

$$g_b(x,w)=\frac{l(x+w)-b(x)}{\sigma^2}w,\qquad x\leftarrow x-\eta g_b(x,w).$$

1. Prove that changing the baseline leaves $\mathbb{E}_w[g_b(x,w)]$ unchanged.
2. Show that this expectation is exactly $\nabla l_\sigma(x)$. You may differentiate under the integral and use Gaussian integration by parts: $\mathbb{E}[w f(w)]=\sigma^2\mathbb{E}[\nabla_w f(w)]$.
3. Use a first-order Taylor expansion and $\mathbb{E}[ww^T]=\sigma^2 I$ to explain why this estimates $\nabla l(x)$ for small $\sigma$. Distinguish the gradient estimate from the parameter change $\Delta x=-\eta g_b$.

Provide your proof in your written submission.

**Problem 11.1.e** [1 pts]: Implement the update from part (d), with $\sigma=0.5$. The baseline is a callable `baseline(x)` returning a scalar. Using `baseline=loss` should recover part (c) for the same sampled perturbation.

In [ ]:
def approximated_gradient_with_baseline(x, rate, baseline):
    """
    Update rule. Receive theta and update it with the next theta.
    Input:
        - x: input variable x.
        - rate: rate of descent, variable "eta".
        - baseline: callable baseline(x) returning a scalar.
    Output:
        - x: updated variable x.
    """

    # YOUR CODE HERE

    return x

As you proved in part (d), the baseline changes the variance but not the mean of the gradient estimator. The choice $b(x)=l(x)$ is useful, especially for small perturbations, but is not generally the variance-minimizing baseline at finite $\sigma$.

For fixed $x$, the scalar baseline minimizing the sum of the component variances is

$$b^*(x)=\frac{\mathbb{E}_w[l(x+w)\|w\|^2]}{\mathbb{E}_w[\|w\|^2]}.$$

Compare `loss(x)`, zero, and 5 using the same initial point and random seed. A baseline far from the nearby loss values can produce much noisier updates. Baselines estimated from data must be independent of the current perturbation (conditional on $x$) for the preceding proof to apply.

In [ ]:
def baseline(x):
    return 5  # Try loss(x) or 0 as well.


def reduced_function(x, rate):
    return approximated_gradient_with_baseline(x, rate, baseline)


np.random.seed(7)
trajectory = gradient_descent(
    0.0025, reduced_function, initial_x=np.array([2.0, -1.0]), iter=10000
)
visualize_trajectory(trajectory)

## How will this notebook be Graded?

If you are enrolled in the class, this notebook will be graded using [Gradescope](www.gradescope.com). You should have gotten the enrollement code on our announcement in Piazza. 

For submission of this assignment, you must do two things. 
- Download and submit the notebook `stochastic_optimization.ipynb` to Gradescope's notebook submission section, along with your notebook for the other problems.
- Write down your answers to 11.1.b, 11.1.d in your PDF submission to Gradescope. 

We will evaluate the local functions in the notebook to see if the function behaves as we have expected. For this exercise, the rubric is as follows:
- [2 pts] 11.1.a must be implemented correctly.
- [1 pts] 11.1.b is answered correctly.
- [2 pts] 11.1.c must be implemented correctly.
- [4 pts] 11.1.d is answered correctly.
- [1 pts] 11.1.e must be implemented correctly.

In [ ]:
from manipulation.exercises.grader import Grader
from manipulation.exercises.rl.test_stochastic_optimization import (
    TestStochasticOptimization,
)

Grader.grade_output([TestStochasticOptimization], [locals()], "results.json")
Grader.print_test_results("results.json")